# Bilevel planning lab — Part 2: build a pyramid 🔺

**On your own now. Still no AI assistants** (those come in Part 3, which is a
local exercise — see the lab README).

Make the planner build a **pyramid**: the **target block resting on top of two
obstructions**.

```
            ┌─────────────┐
            │ target_block│        <- the cap, on top of both
        ┌───┴───┐ ┌───────┴┐
        │  o0   │ │   o1   │       <- the base
   ═════╧═══════╧═╧════════╧═════  table
```

The two obstructions start **apart** on the table. That's the crux: a single "put
the target on top" won't work — think about the *order of operations*: what has
to be true about the obstructions before the cap can go on?

You design the domain, reusing the Part 1 toolkit:
- **Predicates** — classifiers over the state (TODO A) emitted in the
  state abstractor (TODO B);
- **Operators** — lifted preconditions/effects, and **skills** — controllers
  (subclass `MotionPlannedController` like Part 1's place skill) (TODO C);
- the **goal** (TODO D).

Fill the two editable cells (skills, then models). Geometry reminder: a block at
`y` with `height` occupies `[y, y + height]`; `x` is its left edge. `is_on(state,
a, b, {})` is True when `a` rests on `b`; `rectangle_object_to_geom(state, o, {})`
gives a geom with `.vertices` and `.contains_point(px, py)` for corner checks.


In [ ]:
# === Setup: run me first ===
# On Colab: clones the lab and installs the 2D-only packages (no PyBullet;
# ~1-2 min the first time). Locally: just puts the lab on the path.
import os
import subprocess
import sys

REPO_URL = "https://github.com/Princeton-Robot-Planning-and-Learning/kinder-baselines.git"

if "google.colab" in sys.modules:
    if not os.path.exists("/content/kinder-baselines"):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, "/content/kinder-baselines"],
            check=True,
        )
    LAB_DIR = "/content/kinder-baselines/kinder-bilevel-planning/lab"
    try:
        import kinder  # already installed on a warm runtime?
    except ImportError:
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        # Just the environments (no heavy sim backends) + the 2D deps, which
        # include the bilevel_planning planner.
        subprocess.run(pip + ["--no-deps", "kindergarden"], check=True)
        subprocess.run(pip + ["-r", LAB_DIR + "/requirements/lab2d.txt"], check=True)
else:
    LAB_DIR = os.getcwd()
    while LAB_DIR != "/" and not os.path.isdir(
        LAB_DIR + "/kinder-bilevel-planning/lab"
    ):
        LAB_DIR = os.path.dirname(LAB_DIR)
    LAB_DIR += "/kinder-bilevel-planning/lab"

for _p in (LAB_DIR, LAB_DIR + "/notebooks"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import colab_utils  # provided visualization helpers
import kinder

kinder.register_all_environments()
print("\u2705 setup complete")


In [ ]:
# === Provided imports (you do NOT edit this cell) ===
import numpy as np
from bilevel_planning.sesame import run_sesame
from bilevel_planning.structs import (
    LiftedSkill,
    RelationalAbstractGoal,
    RelationalAbstractState,
    SesameModels,
)
from kinder.envs.kinematic2d.object_types import CRVRobotType, RectangleType
from kinder.envs.kinematic2d.obstruction2d import (
    ObjectCentricObstruction2DEnv,
    TargetBlockType,
    TargetSurfaceType,
)
from kinder.envs.kinematic2d.structs import SE2Pose
from kinder.envs.kinematic2d.utils import (
    CRVRobotActionSpace,
    get_suctioned_objects,
    is_on,
    rectangle_object_to_geom,
)
from relational_structs import (
    GroundAtom,
    LiftedAtom,
    LiftedOperator,
    Object,
    Predicate,
    Variable,
)
from relational_structs.spaces import ObjectCentricStateSpace
from crv_skills import (
    MotionPlannedController,
    get_robot_transfer_position,
    make_lifted_controller,
    make_lifted_pick_controller,
)
import colab_utils


## Your skills

Write the place skill(s) the pyramid needs. To make a place-style skill, subclass
`MotionPlannedController` and implement `_target_pose_and_arm(state)` (where to end
up + arm length), `_retract_arm_in_transit()` (`False` while carrying),
`_get_vacuum_actions()` (`(1.0, 0.0)` to hold then release), and
`sample_parameters(state, rng)` (where to release). Return the lifted controllers
from `create_pyramid_controllers`, keyed by name (`pick` is provided).


In [ ]:
# TODO: define your place controller class(es) here, e.g.
#
#   class GroundPlaceNextToController(MotionPlannedController):
#       def __init__(self, objects, action_space, init_constant_state=None):
#           super().__init__(objects, action_space, init_constant_state)
#           self._block = objects[1]
#           ...
#       def _retract_arm_in_transit(self): return False
#       def _get_vacuum_actions(self): return 1.0, 0.0
#       def sample_parameters(self, x, rng): ...
#       def _target_pose_and_arm(self, state): ...


def create_pyramid_controllers(action_space, init_constant_state=None):
    """Return the lifted controllers your skills need (keyed by name)."""
    robot = Variable("?robot", CRVRobotType)
    block = Variable("?block", RectangleType)
    controllers = {
        "pick": make_lifted_pick_controller(action_space, init_constant_state),
    }
    # TODO: add your place controllers, e.g.
    #   controllers["place_next_to"] = make_lifted_controller(
    #       [robot, block, anchor], GroundPlaceNextToController, action_space,
    #       init_constant_state)
    return controllers


## Your models

Define your predicate classifiers, build the `SesameModels` (predicates =
**TODO A**, abstractor logic = **TODO B**, operators + skills = **TODO C**, goal =
**TODO D**), reusing your skills from the cell above. `PickFromTable` is provided
as the pattern to copy.


In [ ]:
# TODO(A): define any predicate classifier helpers you need (like Part 1's
# find_support), e.g. is_adjacent(x, left, right) / is_bridging(x, top, l, r).


def create_pyramid_models(env, num_obstructions=2):
    """Create the planning models for the pyramid task."""
    observation_space = env.observation_space
    action_space = env.action_space
    init_constant_state = getattr(
        env.unwrapped, "_object_centric_env"
    ).initial_constant_state
    sim = ObjectCentricObstruction2DEnv(num_obstructions=num_obstructions)

    def observation_to_state(o):
        return observation_space.devectorize(o)

    def transition_fn(x, u):
        state = x.copy()
        sim.reset(options={"init_state": state})
        obs, _, _, _, _ = sim.step(u)
        return obs.copy()

    types = {CRVRobotType, RectangleType, TargetBlockType, TargetSurfaceType}
    state_space = ObjectCentricStateSpace(types)

    Holding = Predicate("Holding", [CRVRobotType, RectangleType])
    HandEmpty = Predicate("HandEmpty", [CRVRobotType])
    OnTable = Predicate("OnTable", [RectangleType])
    OnTarget = Predicate("OnTarget", [RectangleType])
    # TODO(A): define the predicate(s) the pyramid needs and add them to this set.
    predicates = {Holding, HandEmpty, OnTable, OnTarget}

    def state_abstractor(x):
        robot_o = Object("robot", CRVRobotType)
        target = Object("target_block", TargetBlockType)
        target_surface = Object("target_surface", TargetSurfaceType)
        obstructions = {
            Object(f"obstruction{i}", RectangleType) for i in range(num_obstructions)
        }
        blocks = obstructions | {target}
        atoms = set()
        suctioned_objs = {o for o, _ in get_suctioned_objects(x, robot_o)}
        for obj in suctioned_objs & blocks:
            atoms.add(GroundAtom(Holding, [robot_o, obj]))
        if not suctioned_objs:
            atoms.add(GroundAtom(HandEmpty, [robot_o]))
        for blk in blocks:
            if blk in suctioned_objs:
                continue
            if is_on(x, blk, target_surface, {}):
                atoms.add(GroundAtom(OnTarget, [blk]))
                continue
            # TODO(B): emit your predicate atoms for `blk` (its relationship to the
            # OTHER blocks) instead of always calling it OnTable.
            atoms.add(GroundAtom(OnTable, [blk]))
        return RelationalAbstractState(atoms, {robot_o, target, target_surface} | obstructions)

    def goal_deriver(x):
        # TODO(D): return the goal for the finished pyramid (target supported by
        # both obstructions). Use a predicate you defined.
        del x
        raise NotImplementedError("TODO(D): the pyramid goal")

    robot = Variable("?robot", CRVRobotType)
    block = Variable("?block", RectangleType)
    PickFromTableOperator = LiftedOperator(
        "PickFromTable",
        [robot, block],
        preconditions={LiftedAtom(HandEmpty, [robot]), LiftedAtom(OnTable, [block])},
        add_effects={LiftedAtom(Holding, [robot, block])},
        delete_effects={LiftedAtom(HandEmpty, [robot]), LiftedAtom(OnTable, [block])},
    )
    # TODO(C): define the operator(s) the pyramid needs and pair each with a skill.

    controllers = create_pyramid_controllers(action_space, init_constant_state)
    skills = {
        LiftedSkill(PickFromTableOperator, controllers["pick"]),
        # TODO(C): add LiftedSkill(YourOperator, controllers["your_skill"]) entries.
    }

    return SesameModels(
        observation_space,
        state_space,
        action_space,
        transition_fn,
        types,
        predicates,
        observation_to_state,
        state_abstractor,
        goal_deriver,
        skills,
    )


In [ ]:
ENV_NAME = "kinder/Obstruction2D-o2-v0"
SEED = 0
# All three blocks on the table, none adjacent (matches the local part2 run.py).
LAYOUT = {
    "target_block": (0.15, 0.2, 0.09),
    "obstruction0": (0.4, 0.15, 0.09),
    "obstruction1": (1.1, 0.15, 0.09),
}
ROBOT_X, ROBOT_Y = 0.85, 0.85
TARGET_SURFACE_X = 1.45


def make_pyramid_instance(env, env_models):
    """Reset to the non-adjacent layout; return (initial_state, constant_state)."""
    constant_state = getattr(
        env.unwrapped, "_object_centric_env"
    ).initial_constant_state
    obs, _ = env.reset(seed=SEED)
    state = env_models.observation_to_state(obs).copy()
    robot_o = state.get_object_from_name("robot")
    state.set(robot_o, "x", ROBOT_X); state.set(robot_o, "y", ROBOT_Y)
    state.set(robot_o, "theta", -np.pi / 2)
    for name, (x, w, h) in LAYOUT.items():
        o = state.get_object_from_name(name)
        state.set(o, "x", x); state.set(o, "width", w); state.set(o, "height", h)
    state.set(state.get_object_from_name("target_surface"), "x", TARGET_SURFACE_X)
    return state, constant_state


def plan_pyramid():
    """Build models, plan on the instance; return (plan, constant_state)."""
    env = kinder.make(ENV_NAME)
    env_models = create_pyramid_models(env, num_obstructions=2)
    initial_state, constant_state = make_pyramid_instance(env, env_models)
    plan, _ = run_sesame(
        env_models, initial_state, seed=SEED, max_abstract_plans=5,
        samples_per_step=5, max_skill_horizon=200, timeout=120.0,
    )
    return plan, constant_state


## Done when... (the spec)

This is what "done" means: plan with your models, then check the **geometry** of
the final state — the target rests on top of two obstructions that ended up side
by side as a base. It doesn't care what you named anything.


In [ ]:
plan, constant_state = plan_pyramid()
assert plan is not None, "planner found no plan -- check your predicates/operators/skills/goal"

final = plan.states[-1]
tb = final.get_object_from_name("target_block")
o0 = final.get_object_from_name("obstruction0")
o1 = final.get_object_from_name("obstruction1")


def _span(o):
    return final.get(o, "x"), final.get(o, "x") + final.get(o, "width")


_left, _right = sorted([o0, o1], key=lambda o: final.get(o, "x"))
_left_lo, _left_hi = _span(_left)
_right_lo, _right_hi = _span(_right)
_gap = _right_lo - _left_hi
assert -1e-3 <= _gap <= 0.05, f"obstructions are not adjacent (gap={_gap:.3f})"

_obstruction_top = final.get(_left, "y") + final.get(_left, "height")
assert np.isclose(final.get(tb, "y"), _obstruction_top, atol=2e-3), "target not on top"

_tb_lo = final.get(tb, "x")
_tb_hi = _tb_lo + final.get(tb, "width")
assert _left_lo <= _tb_lo <= _left_hi, "target's left corner is not on the left base"
assert _right_lo <= _tb_hi <= _right_hi, "target's right corner is not on the right base"
print("\u2705 It's a real pyramid!")


## Watch it build

Visualize the solve inline — storyboard then animation.


In [ ]:
colab_utils.show_storyboard(plan.states, constant_state)


In [ ]:
colab_utils.animate_states(plan.states, constant_state)
